In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             confusion_matrix, precision_score, recall_score, f1_score,
                             precision_recall_curve)
import warnings
warnings.filterwarnings('ignore')

ind_hosp = pd.read_parquet('ind_hosp.parquet')
icd_cols = [col for col in ind_hosp.columns if col.startswith('icd_')]
for col in icd_cols:
    n_nan = ind_hosp[col].isna().sum()
    if n_nan > 0:
        ind_hosp[col] = ind_hosp[col].fillna(0)
#ind_hosp = ind_hosp[:1000]

print(f"Dataset: {len(ind_hosp)} rows, {len(ind_hosp.columns)} columns")
print(f"Columns: {ind_hosp.columns.tolist()}")

Dataset: 1823640 rows, 133 columns
Columns: ['subject_id', 'hadm_id', 'dischtime', 'los', 'age', 'day', 'current_date', 'num_diagnoses', 'num_chronic', 'ccsr_FAC021', 'ccsr_FAC025', 'ccsr_END011', 'ccsr_CIR011', 'ccsr_END010', 'ccsr_CIR007', 'ccsr_END003', 'ccsr_CIR019', 'ccsr_DIG004', 'ccsr_CIR017', 'ccsr_CIR008', 'ccsr_BLD003', 'ccsr_EXT027', 'ccsr_GEN002', 'ccsr_END009', 'icd_I10', 'icd_E785', 'icd_K219', 'icd_Z87891', 'icd_I2510', 'icd_N179', 'icd_F329', 'icd_I4891', 'icd_Z7901', 'icd_F419', 'icd_E119', 'icd_E039', 'icd_Z794', 'icd_D649', 'icd_N390', 'comorbidity_score', 'num_procedures_daily', 'cumulative_procedures', 'num_medications_daily', 'cumulative_medications', 'lab_50912_daily', 'lab_51006_daily', 'lab_50931_daily', 'lab_50983_daily', 'lab_50971_daily', 'lab_50902_daily', 'lab_50882_daily', 'lab_50893_daily', 'lab_50868_daily', 'lab_51222_daily', 'lab_51301_daily', 'lab_51265_daily', 'lab_51221_daily', 'lab_51250_daily', 'lab_51277_daily', 'target_readmission_30d', 'prior_

In [5]:
cols_to_drop = [
    'subject_id', 
    'hadm_id', 
    'dischtime', 
    'current_date',
    'target_readmission_30d',
    'los'
]
cols_to_drop = [c for c in cols_to_drop if c in ind_hosp.columns]
X = ind_hosp.drop(columns=cols_to_drop)
y = ind_hosp['target_readmission_30d']

bool_cols = X.select_dtypes(include=['bool']).columns
if len(bool_cols) > 0:
    X[bool_cols] = X[bool_cols].astype(int)

print(f"\nFeatures: {X.shape[1]}, target: {y.name}")
print(f"Readmission rate: {y.mean():.2%}")

patient_target = ind_hosp.groupby('subject_id')['target_readmission_30d'].max().reset_index()
patient_target.columns = ['subject_id', 'has_readmission']

train_val_ids, test_ids = train_test_split(
    patient_target['subject_id'],
    test_size=0.2,
    random_state=42,
    stratify=patient_target['has_readmission']
)

train_ids, val_ids = train_test_split(
    train_val_ids,
    test_size=0.125,
    random_state=42,
    stratify=patient_target[patient_target['subject_id'].isin(train_val_ids)]['has_readmission']
)

train_mask = ind_hosp['subject_id'].isin(train_ids)
val_mask = ind_hosp['subject_id'].isin(val_ids)
test_mask = ind_hosp['subject_id'].isin(test_ids)

train_rate = ind_hosp[train_mask]['target_readmission_30d'].mean()
val_rate = ind_hosp[val_mask]['target_readmission_30d'].mean()
test_rate = ind_hosp[test_mask]['target_readmission_30d'].mean()

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

row_weights = 1 / ind_hosp.loc[train_mask, 'los']
row_weights = row_weights / row_weights.mean()

print(f"Train readmission rate: {y_train.mean():.2%}")
print(f"Val readmission rate: {y_val.mean():.2%}")
print(f"Test readmission rate: {y_test.mean():.2%}")

scale_pos_weight = (1 - y_train.mean()) / y_train.mean()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")


Features: 127, target: target_readmission_30d
Readmission rate: 20.07%
Train readmission rate: 20.01%
Val readmission rate: 20.41%
Test readmission rate: 20.13%
scale_pos_weight: 4.00


In [6]:
from sklearn.linear_model import LogisticRegression
import joblib
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
from sklearn.ensemble import StackingClassifier

base_lasso = LogisticRegression(
    penalty='l1',
    C=0.1,
    solver='saga',
    max_iter=2000,
    tol=1e-4,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

base_lightgbm = lgb.LGBMClassifier(
    objective='binary',
    metric='auc',
    boosting_type='gbdt',
    num_leaves=63,
    max_depth=25,
    min_child_samples=20,
    learning_rate=0.02,
    n_estimators=1000,
    subsample=0.7,
    subsample_freq=1,
    colsample_bytree=0.7,
    reg_alpha=0.1,
    reg_lambda=0.3,
    min_split_gain=0.01,
    scale_pos_weight=scale_pos_weight,
    is_unbalance=False,
    random_state=42,
    verbose=-1,
    n_jobs=-1
)

base_rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    random_state=42,
    max_samples=0.8,
    class_weight='balanced',
    n_jobs=-1
)

meta_model = LogisticRegression(
    penalty='l1',
    C=1.0,
    solver='saga',
    max_iter=1000,
    random_state=42
)

super_learner = StackingClassifier(
    estimators=[
        ('lasso', base_lasso),
        ('lgbm', base_lightgbm),
        ('rf', base_rf)
    ],
    final_estimator=meta_model,
    cv=5,
    stack_method='predict_proba',
    passthrough=False 
)

print("Training Super Learner (Stacking)...")
super_learner.fit(
    X_train, y_train,
    sample_weight=row_weights
)

print("\nEvaluating on validation set...")
y_val_proba_sl = super_learner.predict_proba(X_val)[:, 1]

precisions_sl, recalls_sl, thresholds_sl = precision_recall_curve(y_val, y_val_proba_sl)
f1_scores_sl = 2 * (precisions_sl * recalls_sl) / (precisions_sl + recalls_sl + 1e-8)
best_threshold_sl = thresholds_sl[np.argmax(f1_scores_sl[:-1])]
print(f"Optimal threshold: {best_threshold_sl:.4f}")

y_test_proba_sl = super_learner.predict_proba(X_test)[:, 1]
y_test_pred_sl = (y_test_proba_sl >= best_threshold_sl).astype(int)

auc_roc_sl = roc_auc_score(y_test, y_test_proba_sl)
auc_prc_sl = average_precision_score(y_test, y_test_proba_sl)
brier_sl = brier_score_loss(y_test, y_test_proba_sl)
precision_sl = precision_score(y_test, y_test_pred_sl)
recall_sl = recall_score(y_test, y_test_pred_sl)
f1_sl = f1_score(y_test, y_test_pred_sl)
tn_sl, fp_sl, fn_sl, tp_sl = confusion_matrix(y_test, y_test_pred_sl).ravel()
specificity_sl = tn_sl / (tn_sl + fp_sl) if (tn_sl + fp_sl) > 0 else 0
accuracy_sl = (tp_sl + tn_sl) / (tp_sl + tn_sl + fp_sl + fn_sl)
npv_sl = tn_sl / (tn_sl + fn_sl) if (tn_sl + fn_sl) > 0 else 0

print(f"AUC-ROC: {auc_roc_sl:.4f}")
print(f"AUC-PRC: {auc_prc_sl:.4f}")
print(f"Brier Score: {brier_sl:.4f}")
print(f"\n(Threshold: {best_threshold_sl:.4f}):")
print(f"Accuracy: {accuracy_sl:.4f}")
print(f"Precision: {precision_sl:.4f}")
print(f"Recall: {recall_sl:.4f}")
print(f"Specificity: {specificity_sl:.4f}")
print(f"NPV: {npv_sl:.4f}")
print(f"F1-Score: {f1_sl:.4f}")

model_path = 'super_learner.pkl'
joblib.dump(super_learner, model_path)
print(f"\nModel saved to {model_path}")

Training Super Learner (Stacking)...

Evaluating on validation set...
Optimal threshold: 0.2166
AUC-ROC: 0.7009
AUC-PRC: 0.3888
Brier Score: 0.1461

(Threshold: 0.2166):
Accuracy: 0.6609
Precision: 0.3200
Recall: 0.6086
Specificity: 0.6741
NPV: 0.8724
F1-Score: 0.4195

Model saved to super_learner.pkl
